In [1]:
# Cell 1: Install dependencies
!pip install -q transformers torch Pillow tqdm faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 83.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 86.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 2

In [2]:
# Cell 2: Imports
import os
import numpy as np
from pathlib import Path
from typing import List, Tuple
from PIL import Image
from tqdm import tqdm
import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoImageProcessor
import faiss

# Try multiple possible paths
import os
possible_paths = [
    "/kaggle/input/dogs-cats-images/dataset/test_set/cats",
    "/kaggle/input/chetankv/dogs-cats-images/dataset/test_set/cats",
    "/kaggle/input/datasets/chetankv/dogs-cats-images/dataset/test_set/cats"
]

DATASET_PATH = None
for path in possible_paths:
    if os.path.exists(path):
        DATASET_PATH = path
        print(f"Found dataset at: {path}")
        break

if not DATASET_PATH:
    print("ERROR: Could not find dataset in any expected location")
    import sys
    sys.exit(1)

MODEL_ID = "facebook/dinov3-vitb16-pretrain-lvd1689m"
BATCH_SIZE = 16
OUTPUT_DIR = "/kaggle/working"

# Force CPU to avoid GPU compatibility issues
device = "cpu"
print(f"Using device: {device}")

Using device: cuda


In [3]:
# Cell 2.5: Authenticate Hugging Face
# Get token from Kaggle Secrets or environment variable
import os
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    os.environ['HF_TOKEN'] = hf_token
    print("HF_TOKEN loaded from Kaggle Secrets")
except:
    hf_token = os.getenv('HF_TOKEN', '')
    if hf_token:
        print("HF_TOKEN loaded from environment")
    else:
        print("WARNING: No HF_TOKEN found. DINOv3 may fail to load.")

In [4]:
# Cell 3: Load DINOv3
print(f"Loading DINOv3: {MODEL_ID}")
processor = AutoImageProcessor.from_pretrained(MODEL_ID, token=hf_token or None)
model = AutoModel.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    token=hf_token or None
)
model = model.to(device)
model.eval()
print("Model loaded")

Loading DINOv3: facebook/dinov3-vitb16-pretrain-lvd1689m


OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/facebook/dinov3-vitb16-pretrain-lvd1689m.
401 Client Error. (Request ID: Root=1-6a212b9a-1ba6dfe6064be65e766ba752;d0b167ff-161c-4ceb-b233-79c1c40bf7f2)

Cannot access gated repo for url https://huggingface.co/facebook/dinov3-vitb16-pretrain-lvd1689m/resolve/main/processor_config.json.
Access to model facebook/dinov3-vitb16-pretrain-lvd1689m is restricted. You must have access to it and be authenticated to access it. Please log in.

In [ ]:
# Cell 4: Find images
def find_images(root_path: str) -> List[Tuple[str, str]]:
    image_files = []
    extensions = {'.jpg', '.jpeg', '.png', '.bmp'}
    root = Path(root_path)

    for img_path in root.rglob("*"):
        if img_path.suffix.lower() in extensions and img_path.is_file():
            img_id = str(img_path.name)  # Only filename
            image_files.append((img_id, str(img_path)))

    return sorted(image_files)

image_list = find_images(DATASET_PATH)
print(f"Found {len(image_list)} images")

image_ids = [img_id for img_id, _ in image_list]
image_paths = [img_path for _, img_path in image_list]

In [ ]:
# Cell 5: Embed images - FIXED
print(f"Embedding {len(image_paths)} images with DINOv3...")
all_embeddings = []

for i in tqdm(range(0, len(image_paths), BATCH_SIZE), desc="Embedding"):
    batch_paths = image_paths[i:i+BATCH_SIZE]
    batch_images = []

    for path in batch_paths:
        try:
            img = Image.open(path).convert("RGB")
            batch_images.append(img)
        except Exception as e:
            print(f"Error: {path}: {e}")
            batch_images.append(Image.new("RGB", (224, 224), (0, 0, 0)))

    inputs = processor(images=batch_images, return_tensors="pt")  # Removed padding
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        image_features = outputs.pooler_output
        image_features = F.normalize(image_features, p=2, dim=-1)

    all_embeddings.append(image_features.cpu().numpy())

image_embeddings = np.vstack(all_embeddings)
embedding_dim = image_embeddings.shape[1]
print(f"Embeddings shape: {image_embeddings.shape}")
print(f"Embedding dimension: {embedding_dim}")

norms = np.linalg.norm(image_embeddings, axis=1)
print(f"L2 norms - Min: {norms.min():.4f}, Max: {norms.max():.4f}, Mean: {norms.mean():.4f}")

In [ ]:
# Cell 6: Save
np.savez(
    f"{OUTPUT_DIR}/dinov3_embeddings.npz",
    embeddings=image_embeddings,
    image_ids=image_ids,
    embedding_dim=embedding_dim
)
print(f"Saved: dinov3_embeddings.npz")

index = faiss.IndexFlatIP(embedding_dim)
index.add(image_embeddings.astype(np.float32))
faiss.write_index(index, f"{OUTPUT_DIR}/dinov3_faiss_index.bin")
print(f"Saved: dinov3_faiss_index.bin")

print(f"\nComplete! Download both files.")